In [181]:
import os
import glob
import pickle
import pandas as pd
import numpy as np

from dask.diagnostics import ProgressBar

from arboreto.utils import load_tf_names
from arboreto.algo import grnboost2

from ctxcore.rnkdb import FeatherRankingDatabase as RankingDatabase
from pyscenic.utils import modules_from_adjacencies, load_motifs
from pyscenic.prune import prune2df, df2regulons
from pyscenic.aucell import aucell
import scanpy as sc
import seaborn as sns
from pyscenic.utils import load_motif_annotations
from ctxcore.recovery import aucs as calc_aucs

In [37]:
from pyscenic.transform import (
    DF_META_DATA,
    df2regulons,
    module2features_auc1st_impl,
    modules2df,
    modules2regulons,
)

In [179]:
DATABASES_GLOB="../process/Arid3a/ranking_feather/concat_arid3a_*.genes_vs_tracks.rankings.feather"

In [180]:
db_fnames = glob.glob(DATABASES_GLOB)
dbs = [RankingDatabase(fname=fname, name=name(fname)) for fname in db_fnames]
dbs

[FeatherRankingDatabase(name="concat_arid3a_supp.genes_vs_tracks.rankings"),
 FeatherRankingDatabase(name="concat_arid3a_glue.genes_vs_tracks.rankings")]

In [158]:
dbs2[0].name

'8.25_test_var_glue.genes_vs_tracks.rankings'

In [4]:
db_fnames = glob.glob(DATABASES_GLOB)

In [6]:
def name(fname):
    return os.path.splitext(os.path.basename(fname))[0]
dbs = [RankingDatabase(fname=fname, name=name(fname)) for fname in db_fnames]
dbs

[FeatherRankingDatabase(name="arid3a.genes_vs_tracks.rankings")]

In [9]:
rna = sc.read("../process/scglue/rna_temp.h5ad")

In [14]:
ex_matrix = pd.DataFrame(rna.layers["counts"].toarray())

In [16]:
ex_matrix.index  = rna.obs_names
ex_matrix.columns  = rna.var_names

In [5]:
db_fnames

['../process/Arid3a/glue_regulon/arid3a.genes_vs_tracks.rankings.feather']

In [17]:
tf_names = ["Arid3a"]

In [18]:
adjacencies = grnboost2(ex_matrix, tf_names=tf_names, verbose=True)

preparing dask client


/home/zhangyufeng/miniconda3/envs/scarches/lib/python3.10/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41843 instead
  warnings.warn(


parsing input
creating dask graph
8 partitions
computing dask graph


/home/zhangyufeng/miniconda3/envs/scarches/lib/python3.10/site-packages/distributed/client.py:3162: UserWarning: Sending large graph of size 297.14 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
'WARNING: infer_data failed for target Arid3a' Retry (1/10). Failure caused by ValueError('Cleaned TF matrix is empty, skipping inference of target Arid3a.').
'WARNING: infer_data failed for target Arid3a' Retry (2/10). Failure caused by ValueError('Cleaned TF matrix is empty, skipping inference of target Arid3a.').
'WARNING: infer_data failed for target Arid3a' Retry (3/10). Failure caused by ValueError('Cleaned TF matrix is empty, skipping inference of target Arid3a.').
'WARNING: infer_data failed for target Arid3a' Retry (4/10). Failure caused by ValueError('Cleaned TF matrix is empty, skipping inference of target Arid3a.').
'WARNING: infer_data failed for target Arid3a' Retry (5/10). Failure caused by ValueError('Cleaned TF matr

shutting down client and local cluster
finished


In [109]:
adjacencies2 = pd.read_csv("../process/Arid3a/scenic/8.25_vargene_draft_grn.csv")

In [151]:
modules2 = list(modules_from_adjacencies(adjacencies2, ex_matrix))


2024-08-25 15:30:38,771 - pyscenic.utils - INFO - Calculating Pearson correlations.

2024-08-25 15:30:38,787 - pyscenic.utils - WARNING - Note on correlation calculation: the default behaviour for calculating the correlations has changed after pySCENIC verion 0.9.16. Previously, the default was to calculate the correlation between a TF and target gene using only cells with non-zero expression values (mask_dropouts=True). The current default is now to use all cells to match the behavior of the R verision of SCENIC. The original settings can be retained by setting 'rho_mask_dropouts=True' in the modules_from_adjacencies function, or '--mask_dropouts' from the CLI.
	Dropout masking is currently set to [False].

2024-08-25 15:30:40,312 - pyscenic.utils - INFO - Creating modules.


In [20]:
modules = list(modules_from_adjacencies(adjacencies, ex_matrix))


2024-08-25 13:32:07,499 - pyscenic.utils - INFO - Calculating Pearson correlations.

2024-08-25 13:32:07,511 - pyscenic.utils - WARNING - Note on correlation calculation: the default behaviour for calculating the correlations has changed after pySCENIC verion 0.9.16. Previously, the default was to calculate the correlation between a TF and target gene using only cells with non-zero expression values (mask_dropouts=True). The current default is now to use all cells to match the behavior of the R verision of SCENIC. The original settings can be retained by setting 'rho_mask_dropouts=True' in the modules_from_adjacencies function, or '--mask_dropouts' from the CLI.
	Dropout masking is currently set to [False].

2024-08-25 13:32:08,404 - pyscenic.utils - INFO - Creating modules.


In [22]:
MOTIF_ANNOTATIONS_FNAME = "../process/Arid3a/glue_regulon/glue_arid3a_ctx_annotation.tsv"

In [168]:
MOTIF_ANNOTATIONS_FNAME = "../process/Arid3a/glue_regulon/glue_arid3a_mod_ctx_annotation.tsv"

In [169]:
# Calculate a list of enriched motifs and the corresponding target genes for all modules.
with ProgressBar():
    df = prune2df(dbs2, modules2, MOTIF_ANNOTATIONS_FNAME,num_workers=20,nes_threshold=0)

[########################################] | 100% Completed | 112.28 s


In [173]:
df.columns

MultiIndex([('Enrichment',                   'AUC'),
            ('Enrichment',                   'NES'),
            ('Enrichment', 'MotifSimilarityQvalue'),
            ('Enrichment',   'OrthologousIdentity'),
            ('Enrichment',            'Annotation'),
            ('Enrichment',               'Context'),
            ('Enrichment',           'TargetGenes'),
            ('Enrichment',             'RankAtMax')],
           )

In [174]:
df.to_csv("../process/Arid3a/glue_regulon/8.25_ctx_regulon.csv")

In [166]:
df.iloc[1:50]

Enrichment                                  \
                            AUC       NES MotifSimilarityQvalue   
TF      MotifID                                                   
Ascl1   Ascl1_glue     0.079459  2.842640                   0.0   
Batf    Batf_glue      0.029375  0.335234                   0.0   
Dmrt1   Dmrt1_glue     0.034885  1.084944                   0.0   
Ehf     Ehf_glue       0.031370  0.969743                   0.0   
Erg     Erg_glue       0.028333  0.573016                   0.0   
Hic1    Hic1_glue      0.029773  0.420593                   0.0   
Hoxa10  Hoxa10_glue    0.039524  0.587633                   0.0   
Insm1   Insm1_glue     0.027869  0.265884                   0.0   
Lhx2    Lhx2_glue      0.040862  1.167418                   0.0   
Otx2    Otx2_glue      0.045695  2.192051                   0.0   
Pax3    Pax3_glue      0.035170  1.114507                   0.0   
Pparg   Pparg_glue     0.029512  0.503280                   0.0   
Rfx2    Rfx2_glue      0.042771  1.263975                   0.0   
Rorb    Rorb_glue      0.027895  0.380033                   0.0   
Rorc    Rorc_glue      0.042938  1.920168                   0.0   
Sox14   Sox14_glue     0.027546  0.428505                   0.0   
Tfcp2l1 Tfcp2l1_glue   0.030851  0.650422                   0.0   
Twist1  Twist1_glue    0.032981  0.963288                   0.0   
Ascl1   Ascl1_glue     0.078929  2.709482                   0.0   
Cst6    Cst6_glue      0.026383  0.090442                   0.0   
Ehf     Ehf_glue       0.036138  1.483649                   0.0   
Erg     Erg_glue       0.022250  0.147264                   0.0   
Esr1    Esr1_glue      0.028226  0.238053                   0.0   
Hic1    Hic1_glue      0.042258  0.891707                   0.0   
Insm1   Insm1_glue     0.030112  0.382540                   0.0   
Lhx2    Lhx2_glue      0.033023  0.607470                   0.0   
Nkx3-1  Nkx3-1_glue    0.025000  0.064378                   0.0   
Nr1d2   Nr1d2_glue     0.031443  0.524632                   0.0   
Nr2f1   Nr2f1_glue     0.026867  0.245191                   0.0   
Olig1   Olig1_glue     0.024423  0.007009                   0.0   
Osr2    Osr2_glue      0.035795  0.958965                   0.0   
Pax3    Pax3_glue      0.030194  0.493840                   0.0   
Rfx2    Rfx2_glue      0.031395  0.353908                   0.0   
Rorc    Rorc_glue      0.052605  2.297990                   0.0   
Sox14   Sox14_glue     0.031652  0.581719                   0.0   
Tfcp2l1 Tfcp2l1_glue   0.032136  0.704369                   0.0   
Tfeb    Tfeb_glue      0.033816  0.596652                   0.0   
Twist1  Twist1_glue    0.050505  2.164755                   0.0   
Ascl1   Ascl1_glue     0.062553  2.114967                   0.0   
Cst6    Cst6_glue      0.041373  0.820439                   0.0   
Dmrt1   Dmrt1_glue     0.027451  0.273925                   0.0   
Dmrt3   Dmrt3_glue     0.029216  0.439523                   0.0   
Ebf1    Ebf1_glue      0.031176  0.434796                   0.0   
Ehf     Ehf_glue       0.026863  0.212670                   0.0   
Eomes   Eomes_glue     0.024706  0.084141                   0.0   
Erg     Erg_glue       0.024118  0.214102                   0.0   
Esr1    Esr1_glue      0.047059  1.239382                   0.0   
Evx2    Evx2_glue      0.078929  2.258087                   0.0   
Hic1    Hic1_glue      0.028627  0.388732                   0.0   

                                                       \
                     OrthologousIdentity   Annotation   
TF      MotifID                                         
Ascl1   Ascl1_glue                   1.0  placeholder   
Batf    Batf_glue                    1.0  placeholder   
Dmrt1   Dmrt1_glue                   1.0  placeholder   
Ehf     Ehf_glue                     1.0  placeholder   
Erg     Erg_glue                     1.0  placeholder   
Hic1    Hic1_glue                    1.0  placeholder   
Hoxa10  Hoxa10_glue    

In [30]:
pd.read_feather(DATABASES_GLOB)

,0610038B21Rik,1010001B22Rik,1110002O04Rik,1110006O24Rik,1110035H17Rik,1110046J04Rik,1500002C15Rik,1500026H17Rik,1600002D24Rik,1700001L05Rik,...,Zic4,Zkscan2,Zkscan7,Zmat1,Zmat4,Zpbp,Zscan2,Zswim1,Zswim5,tracks
0,1740,696,778,1740,366,1069,1740,1053,1383,856,...,721,1740,370,306,83,586,968,1740,257,Arid3a_glue


In [29]:
pd.read_table(MOTIF_ANNOTATIONS_FNAME)

,#motif_id,gene_name,motif_similarity_qvalue,orthologous_identity,description
0,Arid3a_glue,Arid3a,0.0,1.0,placeholder
1,Arid3a_supp,Arid3a,0.0,1.0,placeholder


In [35]:
motif_annotations= load_motif_annotations(MOTIF_ANNOTATIONS_FNAME,motif_similarity_fdr = 0.05,orthologous_identity_threshold=0)

In [36]:
motif_annotations

MotifSimilarityQvalue  OrthologousIdentity   Annotation
TF     MotifID                                                             
Arid3a Arid3a_glue                    0.0                  1.0  placeholder
       Arid3a_supp                    0.0                  1.0  placeholder

In [61]:
len(modules[0])

410

In [74]:
df1 = dbs[0].load(modules[0])

In [119]:
df2 = dbs2[0].load(modules2[0])

In [117]:
df2

genes,1110046J04Rik,1700001O22Rik,1700010K24Rik,1700047F07Rik,1700084C06Rik,1700093J21Rik,1810034E14Rik,2200002D01Rik,2810459M11Rik,2900089D17Rik,...,Zfand2a,Zfp341,Zfp42,Zfp747,Zfp809,Zfp934,Zfpm1,Zfyve28,Zpbp,Zswim5
tracks,,,,,,,,,,,,,,,,,,,,,
Arid3a_glue,1069,1740,769,971,1223,1740,71,813,946,1740,...,864,1034,890,1740,520,806,1383,1258,586,257


In [148]:
df1

genes,1700001O22Rik,1700093J21Rik,1810062O18Rik,2200002D01Rik,2900089D17Rik,3110001I22Rik,3830417A13Rik,4833422C13Rik,4933415A04Rik,5730507C01Rik,...,Zfp599,Zfp747,Zfp791,Zfp809,Zfp820,Zfpm1,Zfyve28,Zkscan2,Zswim1,Zswim5
tracks,,,,,,,,,,,,,,,,,,,,,
Arid3a_glue,1740,1740,966,813,1740,801,375,938,1740,1740,...,1740,1740,409,520,1216,1383,1258,1740,1740,257


In [76]:
features, genes, rankings = df1.index.values, df1.columns.values, df1.values

In [81]:
weights = (
        np.asarray([module[gene] for gene in genes])
        if False
        else np.ones(len(genes))
    )

In [98]:
df1

genes,1700001O22Rik,1700093J21Rik,1810062O18Rik,2200002D01Rik,2900089D17Rik,3110001I22Rik,3830417A13Rik,4833422C13Rik,4933415A04Rik,5730507C01Rik,...,Zfp599,Zfp747,Zfp791,Zfp809,Zfp820,Zfpm1,Zfyve28,Zkscan2,Zswim1,Zswim5
tracks,,,,,,,,,,,,,,,,,,,,,
Arid3a_glue,1740,1740,966,813,1740,801,375,938,1740,1740,...,1740,1740,409,520,1216,1383,1258,1740,1740,257


In [131]:
aucs = calc_aucs(df1.transpose(), dbs[0].total_genes, weights, 0.08)

In [133]:
ness = (aucs - aucs.mean()) / aucs.std()

In [144]:
df1.index

Index(['Arid3a_glue'], dtype='object', name='tracks')

In [145]:
df1.columns[ness>3]

Index(['A830018L16Rik', 'Acnat1', 'Adrb2', 'Ankrd22', 'Cgref1', 'Dnm3',
       'Enox2', 'Exd2', 'Il1rapl1', 'Inpp4b', 'Lingo2', 'Nol4', 'Nrcam',
       'Pcdh11x', 'Runx1t1', 'Ryr3', 'Stard13'],
      dtype='object', name='genes')

In [99]:
dbs[0]

FeatherRankingDatabase(name="arid3a.genes_vs_tracks.rankings")

In [140]:

repeat(modules[0].transcription_factor)

repeat('Arid3a')

In [146]:
features

array(['Arid3a_glue'], dtype=object)

In [141]:
zip(repeat(modules[0].transcription_factor), df1.columns)

In [ ]:
repeat(module.transcription_factor), features[enriched_features_idx]

In [182]:
DATABASES_GLOB="../process/Arid3a/ranking_feather/concat_arid3a_*.genes_vs_tracks.rankings.feather"
db_fnames = glob.glob(DATABASES_GLOB)
dbs = [RankingDatabase(fname=fname, name=name(fname)) for fname in db_fnames]
dbs

[FeatherRankingDatabase(name="concat_arid3a_supp.genes_vs_tracks.rankings"),
 FeatherRankingDatabase(name="concat_arid3a_glue.genes_vs_tracks.rankings")]

In [183]:
MOTIF_ANNOTATIONS_FNAME = "../process/Arid3a/glue_regulon/test_var_ctx_annotation.tsv"

In [184]:
df1 = dbs[0].load(modules[0])

In [185]:
df1

genes,1700001O22Rik,1700093J21Rik,1810062O18Rik,2200002D01Rik,2900089D17Rik,3110001I22Rik,3830417A13Rik,4833422C13Rik,4933415A04Rik,5730507C01Rik,...,Zfp599,Zfp747,Zfp791,Zfp809,Zfp820,Zfpm1,Zfyve28,Zkscan2,Zswim1,Zswim5
tracks,,,,,,,,,,,,,,,,,,,,,
Arid3a_1_glue,1740,1740,990,831,1740,782,388,976,1740,1740,...,1740,1740,377,532,1165,1378,1244,1740,1740,266
Arid3a_2_glue,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,...,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085
Arid3a_3_glue,1186,88,1186,1186,1186,39,1186,1186,1186,1186,...,1186,39,1186,1186,1186,232,1186,1186,39,1186
Arid3a_4_glue,1063,1063,1063,1063,1063,1063,1063,1063,1063,1063,...,1063,1063,1063,1063,1063,1063,1063,1063,1063,1063
Arid3a_5_glue,1353,129,1353,1353,1353,122,1353,1353,1353,1353,...,1353,46,1353,1353,122,527,1353,1353,247,229


In [186]:
aucs = calc_aucs(df1, dbs[0].total_genes, weights, 0.08)

In [188]:
df1

genes,1700001O22Rik,1700093J21Rik,1810062O18Rik,2200002D01Rik,2900089D17Rik,3110001I22Rik,3830417A13Rik,4833422C13Rik,4933415A04Rik,5730507C01Rik,...,Zfp599,Zfp747,Zfp791,Zfp809,Zfp820,Zfpm1,Zfyve28,Zkscan2,Zswim1,Zswim5
tracks,,,,,,,,,,,,,,,,,,,,,
Arid3a_1_glue,1740,1740,990,831,1740,782,388,976,1740,1740,...,1740,1740,377,532,1165,1378,1244,1740,1740,266
Arid3a_2_glue,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085,...,1085,1085,1085,1085,1085,1085,1085,1085,1085,1085
Arid3a_3_glue,1186,88,1186,1186,1186,39,1186,1186,1186,1186,...,1186,39,1186,1186,1186,232,1186,1186,39,1186
Arid3a_4_glue,1063,1063,1063,1063,1063,1063,1063,1063,1063,1063,...,1063,1063,1063,1063,1063,1063,1063,1063,1063,1063
Arid3a_5_glue,1353,129,1353,1353,1353,122,1353,1353,1353,1353,...,1353,46,1353,1353,122,527,1353,1353,247,229


In [ ]:
DATABASES_GLOB="../process/Arid3a/ranking_feather/feather_test_short.genes_vs_tracks.rankings.feather"
db_fnames = glob.glob(DATABASES_GLOB)
dbs = [RankingDatabase(fname=fname, name=name(fname)) for fname in db_fnames]
dbs